# EOS — Evolutionary Optimization Strategy

**EvoIR Stage 1 — Population-based loss weight evolution**

Components:
1. **Teacher-Student EMA** — θ_T ← m·θ_T + (1-m)·θ_S with m=0.999
2. **Population Initialization** — 5 individuals (λ1, λ2) on simplex
3. **Fitness Evaluation** — L1 + MS-SSIM composite loss on cached validation
4. **Crossover** — Convex combination of parents
5. **Mutation** — Small perturbation with probability 0.1
6. **Evolution Loop** — 3 generations, select top-3 parents

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import copy
from typing import List, Tuple, Optional
import numpy as np

try:
    from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure
    HAS_TORCHMETRICS = True
except ImportError:
    HAS_TORCHMETRICS = False
    print('Warning: torchmetrics not found. Install with: pip install torchmetrics')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. MS-SSIM Loss Module

In [ ]:
class MS_SSIMLoss(nn.Module):
    """
    Multi-Scale Structural Similarity Loss.
    Loss = 1 - MS_SSIM (lower is better quality).
    """
    def __init__(self, data_range=1.0):
        super().__init__()
        if HAS_TORCHMETRICS:
            self.ms_ssim = MultiScaleStructuralSimilarityIndexMeasure(
                data_range=data_range,
                kernel_size=5,
                betas=(0.5, 0.5, 0.5, 0.5, 0.5)
            )
        else:
            self.ms_ssim = None
    
    def forward(self, pred, target):
        if self.ms_ssim is not None:
            device = pred.device
            self.ms_ssim = self.ms_ssim.to(device)
            return 1 - self.ms_ssim(pred, target)
        else:
            # Fallback: simple L2 loss as approximation
            return F.mse_loss(pred, target)

print('MS_SSIMLoss defined')

## 2. Population Initialization

In [ ]:
def init_population(size: int = 5) -> List[List[float]]:
    """
    Initialize a population of loss weight individuals.
    Each individual is (λ1, λ2) where λ1 + λ2 = 1.0 (simplex constraint).
    λ1 controls fidelity loss (L1), λ2 controls perceptual loss (MS-SSIM).
    
    Args:
        size: Population size (default: 5)
    Returns:
        List of [λ1, λ2] pairs
    """
    population = []
    for _ in range(size):
        lambda1 = random.uniform(0.1, 0.9)
        lambda2 = 1.0 - lambda1
        population.append([lambda1, lambda2])
    return population

# Demo
pop = init_population(5)
print('Initial population:')
for i, ind in enumerate(pop):
    print(f'  Individual {i}: λ_L1={ind[0]:.3f}, λ_SSIM={ind[1]:.3f}, sum={sum(ind):.3f}')

## 3. Fitness Evaluation

In [ ]:
def evaluate_individual_from_cache(
    individual: List[float],
    restored_list: List[torch.Tensor],
    clean_list: List[torch.Tensor],
    l1_loss: nn.Module,
    ms_ssim_loss: nn.Module
) -> float:
    """
    Evaluate fitness of an individual on cached restored/clean pairs.
    Uses FROZEN network outputs (no gradient computation).
    
    Fitness = -total_loss (higher is better for selection).
    
    Args:
        individual: [λ1, λ2] loss weights
        restored_list: Cached model outputs
        clean_list: Corresponding ground truth
        l1_loss: L1 loss function
        ms_ssim_loss: MS-SSIM loss function
    Returns:
        Negative total loss (higher = better)
    """
    total_loss = 0.0
    lambda1, lambda2 = individual
    
    with torch.no_grad():
        for restored, clean in zip(restored_list, clean_list):
            if restored.dim() == 3:
                restored = restored.unsqueeze(0)
                clean = clean.unsqueeze(0)
            loss = lambda1 * l1_loss(restored, clean) + lambda2 * ms_ssim_loss(restored, clean)
            total_loss += loss.item()
    
    return -total_loss  # Negate: we want to MINIMIZE loss but MAXIMIZE fitness

print('evaluate_individual_from_cache defined')

## 4. Crossover & Mutation Operators

In [ ]:
def crossover(p1: List[float], p2: List[float]) -> Tuple[List[float], List[float]]:
    """
    Convex combination crossover on simplex.
    child = α·parent1 + (1-α)·parent2, maintaining λ1+λ2≈1.
    
    Args:
        p1, p2: Parent individuals [λ1, λ2]
    Returns:
        Two child individuals
    """
    alpha = random.random()
    child1 = [
        alpha * p1[0] + (1 - alpha) * p2[0],
        alpha * p1[1] + (1 - alpha) * p2[1]
    ]
    child2 = [
        alpha * p2[0] + (1 - alpha) * p1[0],
        alpha * p2[1] + (1 - alpha) * p1[1]
    ]
    return child1, child2


def mutate(individual: List[float], mutation_rate: float = 0.1) -> List[float]:
    """
    Mutate individual with small perturbation, keeping on simplex.
    
    With probability `mutation_rate`:
    - Add random δ ∈ [-0.1, 0.1] to λ1
    - Clamp λ1 to [0.1, 0.9]
    - Set λ2 = 1 - λ1 (maintain simplex constraint)
    
    Args:
        individual: [λ1, λ2]
        mutation_rate: Probability of mutation
    Returns:
        Possibly mutated individual
    """
    if random.random() < mutation_rate:
        delta = random.uniform(-0.1, 0.1)
        individual[0] = min(max(individual[0] + delta, 0.1), 0.9)
        individual[1] = 1.0 - individual[0]
    return individual

print('Crossover and Mutation operators defined')

## 5. Full EOS Evolution Loop

In [ ]:
def evolve_loss_weights_from_cache(
    restored_list: List[torch.Tensor],
    clean_list: List[torch.Tensor],
    l1_loss: nn.Module,
    ms_ssim_loss: nn.Module,
    generations: int = 3,
    pop_size: int = 5
) -> List[float]:
    """
    Full EOS: Evolve optimal loss weights (λ1, λ2) using cached outputs.
    
    Algorithm:
    1. Initialize random population on simplex
    2. For each generation:
       a. Evaluate all individuals on cached data
       b. Select top-3 as parents
       c. Produce children via crossover + mutation
       d. Keep best individual (elitism)
    3. Return best individual found
    
    Args:
        restored_list: Cached model outputs from recent batches
        clean_list: Corresponding ground truth
        l1_loss: L1 loss function
        ms_ssim_loss: MS-SSIM loss function
        generations: Number of evolution generations (default: 3)
        pop_size: Population size (default: 5)
    Returns:
        Best [λ1, λ2] found
    """
    population = init_population(pop_size)
    best = None
    best_score = -float('inf')
    
    for gen in range(generations):
        # Evaluate fitness
        scores = []
        for ind in population:
            score = evaluate_individual_from_cache(
                ind, restored_list, clean_list, l1_loss, ms_ssim_loss
            )
            scores.append((score, ind))
        
        # Sort by fitness (highest first)
        scores.sort(reverse=True)
        gen_best_score, gen_best = scores[0]
        
        if gen_best_score > best_score:
            best_score = gen_best_score
            best = gen_best[:]
        
        # Build next generation
        next_population = [gen_best[:]]  # Elitism: keep the best
        
        while len(next_population) < pop_size:
            # Select 2 parents from top-3
            p1, p2 = random.sample(scores[:3], 2)
            child1, child2 = crossover(p1[1], p2[1])
            next_population.append(mutate(child1[:]))
            next_population.append(mutate(child2[:]))
        
        population = next_population[:pop_size]
    
    return best

print('evolve_loss_weights_from_cache defined')

## 6. Teacher-Student EMA Framework

In [ ]:
class EMATeacher:
    """
    Exponential Moving Average Teacher for teacher-student framework.
    
    The teacher network is a slow-moving average of the student:
    θ_T ← momentum * θ_T + (1 - momentum) * θ_S
    
    Used for stable evaluation during EOS evolution.
    
    Args:
        student_model: The student (training) model
        momentum: EMA decay rate (default: 0.999)
    """
    def __init__(self, student_model: nn.Module, momentum: float = 0.999):
        self.momentum = momentum
        self.teacher_model = copy.deepcopy(student_model)
        
        # Freeze teacher parameters
        for param in self.teacher_model.parameters():
            param.requires_grad = False
        
        self.teacher_model.eval()
    
    @torch.no_grad()
    def update(self, student_model: nn.Module):
        """
        Update teacher weights with EMA of student weights.
        Call after each optimizer step.
        """
        for t_param, s_param in zip(
            self.teacher_model.parameters(),
            student_model.parameters()
        ):
            t_param.data.mul_(self.momentum).add_(
                s_param.data, alpha=1.0 - self.momentum
            )
    
    @torch.no_grad()
    def evaluate(self, x: torch.Tensor) -> torch.Tensor:
        """Run teacher model in eval mode (frozen, no gradients)."""
        self.teacher_model.eval()
        return self.teacher_model(x)
    
    def state_dict(self):
        return self.teacher_model.state_dict()
    
    def load_state_dict(self, state_dict):
        self.teacher_model.load_state_dict(state_dict)

print('EMATeacher defined')

## 7. Complete EOS Manager

In [ ]:
class EOSManager:
    """
    Complete EOS (Evolutionary Optimization Strategy) manager.
    
    Integrates:
    - Teacher-Student EMA
    - Periodic evolution trigger (every T iterations)
    - Loss weight tracking and logging
    
    Args:
        student_model: Training model
        ema_momentum: EMA decay (default: 0.999)
        evolution_interval: Trigger evolution every T steps (default: 500)
        pop_size: Population size (default: 5)
        generations: Evolution generations (default: 3)
        initial_lambda: Starting loss weights [λ_L1, λ_SSIM]
    """
    def __init__(
        self,
        student_model: nn.Module,
        ema_momentum: float = 0.999,
        evolution_interval: int = 500,
        pop_size: int = 5,
        generations: int = 3,
        initial_lambda: Optional[List[float]] = None
    ):
        self.ema_teacher = EMATeacher(student_model, ema_momentum)
        self.evolution_interval = evolution_interval
        self.pop_size = pop_size
        self.generations = generations
        
        # Current loss weights
        if initial_lambda is None:
            initial_lambda = [0.8, 0.2]
        self.lambda_l1 = initial_lambda[0]
        self.lambda_ssim = initial_lambda[1]
        
        # Loss functions
        self.l1_loss = nn.L1Loss()
        self.ms_ssim_loss = MS_SSIMLoss(data_range=1.0)
        
        # Cache for evolution
        self.restored_cache = []
        self.clean_cache = []
        self.step_count = 0
        
        # History
        self.weight_history = [(self.lambda_l1, self.lambda_ssim)]
    
    def compute_loss(
        self, restored: torch.Tensor, clean: torch.Tensor
    ) -> torch.Tensor:
        """Compute weighted composite loss."""
        loss = (
            self.lambda_l1 * self.l1_loss(restored, clean) +
            self.lambda_ssim * self.ms_ssim_loss(restored, clean)
        )
        return loss
    
    def step(
        self,
        student_model: nn.Module,
        restored: torch.Tensor,
        clean: torch.Tensor
    ) -> bool:
        """
        Call after each training step.
        
        - Updates EMA teacher
        - Caches outputs for evolution
        - Triggers evolution every `evolution_interval` steps
        
        Args:
            student_model: Current student model
            restored: Model output (detached)
            clean: Ground truth (detached)
        Returns:
            True if evolution was triggered this step
        """
        self.step_count += 1
        
        # Update EMA teacher
        self.ema_teacher.update(student_model)
        
        # Cache for evolution
        self.restored_cache.append(restored.detach().cpu())
        self.clean_cache.append(clean.detach().cpu())
        
        # Keep cache bounded
        max_cache = 32
        if len(self.restored_cache) > max_cache:
            self.restored_cache = self.restored_cache[-max_cache:]
            self.clean_cache = self.clean_cache[-max_cache:]
        
        # Evolution trigger
        if self.step_count % self.evolution_interval == 0:
            self._evolve()
            return True
        
        return False
    
    def _evolve(self):
        """Run evolutionary search on cached data."""
        if len(self.restored_cache) < 2:
            return
        
        new_weights = evolve_loss_weights_from_cache(
            self.restored_cache,
            self.clean_cache,
            self.l1_loss,
            self.ms_ssim_loss,
            generations=self.generations,
            pop_size=self.pop_size
        )
        
        self.lambda_l1, self.lambda_ssim = new_weights
        self.weight_history.append((self.lambda_l1, self.lambda_ssim))
        
        # Clear cache after evolution
        self.restored_cache = []
        self.clean_cache = []
    
    def get_weights(self) -> Tuple[float, float]:
        return self.lambda_l1, self.lambda_ssim
    
    def get_history(self) -> List[Tuple[float, float]]:
        return self.weight_history

print('EOSManager defined')

## 8. Verification Tests

In [ ]:
print('=== Test 1: Population Initialization ===')
pop = init_population(5)
for ind in pop:
    assert 0.1 <= ind[0] <= 0.9, f'λ1 out of range: {ind[0]}'
    assert abs(ind[0] + ind[1] - 1.0) < 1e-6, f'Sum != 1: {sum(ind)}'
print('  PASSED: All individuals on simplex')

print('\n=== Test 2: Crossover ===')
p1, p2 = [0.3, 0.7], [0.8, 0.2]
for _ in range(10):
    c1, c2 = crossover(p1, p2)
    assert abs(c1[0] + c1[1] - 1.0) < 1e-6
    assert abs(c2[0] + c2[1] - 1.0) < 1e-6
print('  PASSED: Crossover maintains simplex')

print('\n=== Test 3: Mutation ===')
for _ in range(100):
    ind = mutate([0.5, 0.5], mutation_rate=1.0)
    assert 0.1 <= ind[0] <= 0.9
    assert abs(ind[0] + ind[1] - 1.0) < 1e-6
print('  PASSED: Mutation stays in bounds')

print('\n=== Test 4: Evolution on synthetic data ===')
torch.manual_seed(42)
restored_list = [torch.randn(1, 3, 64, 64) for _ in range(5)]
clean_list = [torch.randn(1, 3, 64, 64) for _ in range(5)]
l1 = nn.L1Loss()
msssim = MS_SSIMLoss()
best = evolve_loss_weights_from_cache(restored_list, clean_list, l1, msssim)
print(f'  Best weights: λ_L1={best[0]:.4f}, λ_SSIM={best[1]:.4f}')
assert 0.1 <= best[0] <= 0.9
assert abs(best[0] + best[1] - 1.0) < 1e-6
print('  PASSED: Evolution produces valid weights')

print('\n=== Test 5: EMA Teacher ===')
# Create simple model for testing
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 3, 3, padding=1)
    def forward(self, x):
        return self.conv(x)

student = SimpleModel()
ema = EMATeacher(student, momentum=0.999)
initial_teacher_params = {n: p.clone() for n, p in ema.teacher_model.named_parameters()}

# Simulate training step
optim = torch.optim.SGD(student.parameters(), lr=0.1)
x = torch.randn(1, 3, 16, 16)
out = student(x)
out.mean().backward()
optim.step()
ema.update(student)

# Check teacher params moved
for name, param in ema.teacher_model.named_parameters():
    diff = (param - initial_teacher_params[name]).abs().max().item()
    if diff > 0:
        print(f'  Teacher param {name} moved by {diff:.6f}')
        break
print('  PASSED: EMA teacher updates correctly')

print('\n=== Test 6: EOSManager ===')
manager = EOSManager(student, evolution_interval=5)
for i in range(10):
    restored = torch.randn(1, 3, 64, 64)
    clean = torch.randn(1, 3, 64, 64)
    evolved = manager.step(student, restored, clean)
    if evolved:
        w = manager.get_weights()
        print(f'  Step {i+1}: Evolution triggered! λ_L1={w[0]:.4f}, λ_SSIM={w[1]:.4f}')
print(f'  Weight history ({len(manager.get_history())} entries): {manager.get_history()}')
print('  PASSED')

print('\n All EOS tests passed!')